<a href="https://colab.research.google.com/github/msiplab/VieWork/blob/master/code/python/vie_sec07_fourier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://matlab.mathworks.com/open/github/v1?repo=msiplab/VieWork&file=code/matlab/vie_sec07_fourier.m&project=VieWork.prj" target="_parent"><img src="https://www.mathworks.com/images/responsive/global/open-in-matlab-online.svg" alt="Open in MATLAB Online"/></a>

# 第7回 フーリエ解析
村松正吾　「画像情報工学」

動作確認：Python 3.12（Google Colab），NumPy，SciPy，scikit-image，Matplotlib

MATLAB のライブスクリプト `code/matlab/vie_sec07_fourier.m` を Python に移植したもの。第7回のスライド（vie2026-07）の図と数値を計算する。多次元信号の周波数，多次元周期信号と周期構造行列，フーリエ解析（FT, DSFT, DFT），DFT と DSFT の関係，DFT の分離処理と行列表現，FFT の演算量，周波数スペクトル，周波数応答，位相応答の重要性を扱う。

スライドの図と数値は MATLAB 版で作っている。新潟大学の学生は[大学の包括ライセンス](https://jp.mathworks.com/academia/tah-portal/niigata-university-31576302.html)で MATLAB を使えるので，MATLAB 版もぜひ動かしてほしい。上の「Open in MATLAB Online」のボタンを押すと，ブラウザだけで MATLAB Online が開き，VieWork が自分の MATLAB Drive に複製されて，このノートブックに対応するライブスクリプトが開く（インストールは不要）。さらに [MATLAB Connector](https://jp.mathworks.com/products/matlab-drive.html) を手元の PC に入れると，MATLAB Drive のファイルが PC のフォルダと自動で同期されるので，MATLAB Online で作業したファイルを手元の MATLAB でもそのまま開ける。

記号は教科書に合わせる。連続変数の角周波数ベクトルを $\boldsymbol{\nu}$，離散変数の正規化角周波数ベクトルを $\boldsymbol{\omega}=(\omega_1\ \omega_2)^\top$，周期構造行列を $\boldsymbol{N}$，標本化行列を $\boldsymbol{L}$ とし，DSFT を $X(\mathrm{e}^{\mathrm{j}\boldsymbol{\omega}^\top})$ と書く。次元の添え字は 1 始まりで，第 1 次元（$n_1$，$\omega_1$）が垂直，第 2 次元（$n_2$，$\omega_2$）が水平。Python の配列では第 1 次元が軸 0（行），第 2 次元が軸 1（列）に対応する。

画像処理には scikit-image，SciPy，NumPy を使う。教科書の例題の数値はスライドと一致することを `assert` で確かめている。画像から計算する値は，ライブラリの実装が MATLAB と異なるため，スライドの値とわずかに異なることがある。

## 準備
共通ヘルパー `vie.py`（サンプル画像の取得，日本語フォントと配色の設定）を読み込む。Colab では GitHub から取得する。

In [ ]:
import importlib
import os
import sys
import urllib.request

# Colab では毎回 GitHub から最新の vie.py を取得する（VieWork を clone した場合は同じフォルダの vie.py を使う）
if "google.colab" in sys.modules or not os.path.exists("vie.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/msiplab/VieWork/master/code/python/vie.py", "vie.py")

from fractions import Fraction

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.mplot3d.art3d import Line3DCollection, Poly3DCollection
from scipy import signal
from skimage.util import img_as_float
import vie

vie = importlib.reload(vie)   # 取得し直した vie.py を読み込み直す

# 図の配色はスライドと揃え，ロゴの 3 色（メインの緑，寒色系の青，暖色系の橙）と灰色を使う
cmain, ccool, cwarm, cgray = vie.CMAIN, vie.CCOOL, vie.CWARM, vie.CGRAY


def show(ax, X, title=None):
    """画像を軸なし・縦横比 1 で表示する（グレースケールは 0 を黒，最大値（255 か 1）を白）．"""
    vmax = 255 if X.dtype == np.uint8 else 1
    ax.imshow(X, cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_axis_off()
    if title is not None:
        ax.set_title(title)

振幅の曲面には淡い緑から濃い緑への色図を使う。網目の線を高さに応じた色で描く関数と，π の有理数倍・有理数を文字列にする関数も用意しておく。

In [ ]:
cm = np.array(cmain)
cmapMain = LinearSegmentedColormap.from_list("main", [1 - 0.35 * (1 - cm), cm, 0.6 * cm])   # 淡い緑→緑→濃い緑


def mesh(ax, X, Y, Z, cmap, lw=0.8):
    """網目の線を高さに応じた色で描く（曲面の網目表示）"""
    P = np.stack([X, Y, Z], axis=-1)
    segs = np.concatenate([np.stack([P[:, :-1], P[:, 1:]], axis=2).reshape(-1, 2, 3),
                           np.stack([P[:-1], P[1:]], axis=2).reshape(-1, 2, 3)])
    lc = Line3DCollection(segs, cmap=cmap, lw=lw)
    lc.set_array(segs[:, :, 2].mean(axis=1))
    lc.set_clim(Z.min(), Z.max())
    ax.add_collection3d(lc)
    ax.set(xlim=(X.min(), X.max()), ylim=(Y.min(), Y.max()), zlim=(Z.min(), Z.max()))
    return lc


def pistr(v):
    """π の有理数倍を文字列にする（例：3*pi/4 → "3π/4"，0 → "0"）"""
    f = Fraction(v / np.pi).limit_denominator(1000)
    p, q = f.numerator, f.denominator
    if p == 0:
        return "0"
    s = ("-" if p < 0 else "") + ("π" if abs(p) == 1 else f"{abs(p)}π")
    return s if q == 1 else s + f"/{q}"


def fracstr(v):
    """有理数を "p/q" の文字列にする（整数ならそのまま）"""
    f = Fraction(v).limit_denominator(1000)
    return f"{f.numerator}" if f.denominator == 1 else f"{f.numerator}/{f.denominator}"

## 一次元余弦波の角周波数
$x(t)=\cos(\nu t)$ の角周波数 $\nu$ は位相 $\nu t$ の時間微分。単位時間に 3 周期の波なら $\nu = 2\pi\times 3 = 6\pi$ [rad/s]，周期は $2\pi/\nu=1/3$。

In [ ]:
t = np.linspace(0, 1, 1001)
nu = 2 * np.pi * 3
T1 = 2 * np.pi / nu
print("ν =", nu, "=", pistr(nu), "，周期 =", T1, "=", fracstr(T1))
assert pistr(nu) == "6π" and fracstr(T1) == "1/3"   # スライドの値

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.2), layout="constrained")
ax.plot(t, np.cos(nu * t), color=cmain, lw=2.4)
ax.set(ylim=(-1.2, 1.2), xlabel="$t$", ylabel="$x(t)$",
       title="$x(t)=\\cos(6\\pi t)$（単位時間に 3 周期）")
ax.grid(True)
plt.show()

## 信号は正弦波の足し合わせ
3 つの余弦波（1, 3, 7 周期／単位時間，振幅 1, 0.5, 0.25）の和と，その振幅スペクトル（周波数分布）。

In [ ]:
f = np.array([1, 3, 7]); A = np.array([1, 0.5, 0.25])
comps = A[:, None] * np.cos(2 * np.pi * f[:, None] * t)
xsum = comps.sum(axis=0)
compColors = [ccool, cmain, cwarm]
ttl = ["$\\cos 2\\pi t$", "$0.5\\cos 6\\pi t$", "$0.25\\cos 14\\pi t$"]
fig, axs = plt.subplots(4, 1, figsize=(4.5, 4.4), layout="constrained")
for k in range(3):
    axs[k].plot(t, comps[k], color=compColors[k], lw=1.5)
    axs[k].set_ylim(-1.1, 1.1)
    axs[k].set_title(ttl[k], color=compColors[k], fontsize=12)
    axs[k].axis("off")
axs[3].plot(t, xsum, "k", lw=2)
axs[3].set_title("和 $x(t)$", fontsize=12)
axs[3].axis("off")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.6), layout="constrained")
ml, sl, bl = ax.stem(2 * np.pi * f, A, basefmt=" ")
plt.setp(ml, color=cmain); plt.setp(sl, color=cmain, lw=2)
ax.set(xlim=(0, 2 * np.pi * 8), ylim=(0, 1.1), xticks=2 * np.pi * f,
       xticklabels=["$2\\pi$", "$6\\pi$", "$14\\pi$"], xlabel="角周波数 $\\nu$", ylabel="振幅")
ax.grid(True)
plt.show()

## 例題：多次元余弦波の標本化
教科書の例題。$D$ 次元余弦波 $u(\boldsymbol{q})=\cos(\boldsymbol{\nu}^\top\boldsymbol{q})$ をディラックのデルタで，標本化行列 $\boldsymbol{L}=\mathrm{diag}(\Delta_1,\ldots,\Delta_D)$ の格子上で標本化すると

$$x[\boldsymbol{n}]=\cos(\boldsymbol{\nu}^\top\boldsymbol{L}\boldsymbol{n})=\cos(\boldsymbol{\omega}^\top\boldsymbol{n}),\quad \boldsymbol{\omega}=\boldsymbol{L}^\top\boldsymbol{\nu}$$

すなわち $\omega_d=\Delta_d\nu_d=\nu_d/F_d$（$F_d=\Delta_d^{-1}$：標本化周波数）。数値例として，教科書 図 4.2 (c) の $\boldsymbol{\nu}=(4\pi\ 2\pi)^\top$ を $\Delta_1=\Delta_2=1/8$（$F_d=8$）で標本化する。

In [ ]:
nuS = np.array([4 * np.pi, 2 * np.pi])             # 角周波数ベクトル ν（図 4.2 (c)）
L = np.diag([1 / 8, 1 / 8])                        # 標本化行列 L
omegaS = L.T @ nuS                                 # 正規化角周波数ベクトル ω = L^T ν
print("ω =", omegaS)
n2g, n1g = np.meshgrid(np.arange(16), np.arange(16))   # 標本点の添え字 n = (n1 n2)^T
nS = np.vstack([n1g.ravel(), n2g.ravel()])
errS = np.max(np.abs(np.cos(nuS @ L @ nS) - np.cos(omegaS @ nS)))   # 両者が一致することを確認
print("errS =", errS)
snu = "  ".join(pistr(v) for v in nuS)
sL = ",".join(fracstr(v) for v in np.diag(L))
som = "  ".join(pistr(v) for v in omegaS)
print("ν =", snu, "，L の対角 =", sL, "，ω =", som)
assert (snu, sL, som) == ("4π  2π", "1/8,1/8", "π/2  π/4")   # スライドの値

## 例題：非長方周期構造と基本周期内の整数ベクトル
教科書の例題（図 4.3）。基本周期（灰色の平行四辺形）の辺 $\boldsymbol{n}_1=[2\ 0]^\top$，$\boldsymbol{n}_2=[1\ 2]^\top$ を列に並べると周期構造行列 $\boldsymbol{N}=[\boldsymbol{n}_1\ \boldsymbol{n}_2]$ が得られる。

In [ ]:
n1v = np.array([2, 0])
n2v = np.array([1, 2])
Nmat = np.column_stack([n1v, n2v])                 # 周期構造行列 N
detN = np.linalg.det(Nmat)
print(Nmat)
print("det(N) =", detN)
assert np.array_equal(Nmat, [[2, 1], [0, 2]])      # スライドの値

基本周期 $\mathrm{FPD}(\boldsymbol{N})=\{\boldsymbol{N}\boldsymbol{r}\mid\boldsymbol{r}\in[0,1)^2\}$ に含まれる整数ベクトルの集合 $\mathcal{N}(\boldsymbol{N})$ を，平行四辺形を囲む長方形内の整数ベクトル $\boldsymbol{n}$ について $\boldsymbol{r}=\boldsymbol{N}^{-1}\boldsymbol{n}\in[0,1)^2$ を調べて求める。要素数は $|\det(\boldsymbol{N})|$ に一致する。

In [ ]:
corners = Nmat @ np.array([[0, 1, 0, 1], [0, 0, 1, 1]])   # 平行四辺形の 4 頂点
c2, c1 = np.meshgrid(np.arange(np.floor(corners[1].min()), np.ceil(corners[1].max()) + 1),
                     np.arange(np.floor(corners[0].min()), np.ceil(corners[0].max()) + 1))
cand = np.vstack([c1.ravel(), c2.ravel()])         # 候補の整数ベクトル
r = np.linalg.solve(Nmat, cand)                    # r = N^{-1} n
tol = 1e-12
inFPD = np.all((r > -tol) & (r < 1 - tol), axis=0)
sel = cand[:, inFPD]
setN = sel[:, np.lexsort((sel[1], sel[0]))].astype(int)   # 基本周期内の整数ベクトル（列）
cardN = setN.shape[1]                              # 要素数 |N(N)|
print(setN)
print("|N(N)| =", cardN)
assert cardN == round(abs(detN))                   # |N(N)| = |det(N)| を確認
assert np.array_equal(setN, [[0, 1, 1, 2], [0, 0, 1, 1]])   # スライドの値

## 例題：二次元配列の DSFT と DFT
$x[0,0]=x[1,0]=x[0,1]=x[1,1]=1$，その他 0。DSFT は

$$X(\mathrm{e}^{\mathrm{j}\boldsymbol{\omega}^\top})=(1+\mathrm{e}^{-\mathrm{j}\omega_1})(1+\mathrm{e}^{-\mathrm{j}\omega_2})=4\cos\frac{\omega_1}{2}\cos\frac{\omega_2}{2}\mathrm{e}^{-\mathrm{j}(\omega_1+\omega_2)/2}$$

周期構造行列 $\boldsymbol{N}=\mathrm{diag}(2,2)$ の DFT は DSFT を $\omega_d=\pi k_d$ で標本化したもの。行の添え字が $k_1$，列の添え字が $k_2$。

In [ ]:
x22 = np.ones((2, 2))
X22 = np.fft.fft2(x22)
print(X22)


def dsft(w1, w2):
    return 4 * np.cos(w1 / 2) * np.cos(w2 / 2) * np.exp(-1j * (w1 + w2) / 2)


Xs = np.array([[dsft(0, 0), dsft(0, np.pi)], [dsft(np.pi, 0), dsft(np.pi, np.pi)]])
maxerr = np.max(np.abs(Xs - X22))                  # 一致を確認（丸め誤差程度）
print("maxerr =", maxerr)
assert np.array_equal(np.real(X22), [[4, 0], [0, 0]])   # スライドの値

DSFT の閉じた形を，定義の和と任意の周波数で比べておく。

In [ ]:
rng = np.random.default_rng()
wt = 2 * np.pi * rng.random((2, 5)) - np.pi
n2e, n1e = np.meshgrid(np.arange(2), np.arange(2))
Xdef = np.array([np.sum(x22 * np.exp(-1j * (wt[0, k] * n1e + wt[1, k] * n2e))) for k in range(5)])
errDSFT = np.max(np.abs(Xdef - dsft(wt[0], wt[1])))
print("errDSFT =", errDSFT)

**分離処理**：対角の周期構造行列では，まず第 1 次元（$n_1$ 方向：各列）に一次元 DFT を施して $X^{\{1\}}[k_1;n_2]$ を求め，次に第 2 次元（$n_2$ 方向：各行）に一次元 DFT を施す。

In [ ]:
X1sep = np.fft.fft(x22, axis=0)                    # X^{1}[k1; n2]（各列の DFT）
Xsep = np.fft.fft(X1sep, axis=1)                   # X[k1, k2]（各行の DFT）
print(X1sep)
print(Xsep)
assert np.max(np.abs(Xsep - X22)) < 1e-12
assert np.array_equal(np.real(X1sep), [[2, 2], [0, 0]])   # スライドの値

振幅スペクトル $|X(\mathrm{e}^{\mathrm{j}\boldsymbol{\omega}^\top})|$ の曲面と，DFT の標本 $|X[k_1,k_2]|$（$\boldsymbol{\omega}=\pi\boldsymbol{k}$）を重ねて描く。

In [ ]:
W2, W1 = np.meshgrid(np.linspace(-np.pi, np.pi, 41), np.linspace(-np.pi, np.pi, 41))   # 横軸 ω2，縦軸 ω1
fig = plt.figure(figsize=(5.5, 4.5), layout="constrained")
ax = fig.add_subplot(projection="3d")
mesh(ax, W2, W1, np.abs(dsft(W1, W2)), cmapMain)
K2, K1 = np.meshgrid(np.arange(2), np.arange(2))
for k2, k1, v in zip(np.pi * K2.ravel(), np.pi * K1.ravel(), np.abs(X22).ravel()):
    ax.plot([k2, k2], [k1, k1], [0, v], color=cwarm, lw=2.5)
    ax.scatter([k2], [k1], [v], color=cwarm, s=50, depthshade=False)
ax.view_init(elev=30, azim=-125)
pit = [-np.pi, 0, np.pi]; pil = ["$-\\pi$", "$0$", "$\\pi$"]
ax.set(xticks=pit, xticklabels=pil, yticks=pit, yticklabels=pil, xlabel="$\\omega_2$", ylabel="$\\omega_1$",
       zlabel="$|X(\\mathrm{e}^{\\mathrm{j}\\boldsymbol{\\omega}^\\top})|$")
ax.set_box_aspect(None, zoom=0.85)                 # 軸ラベルが切れないよう少し小さく描く
plt.show()

## DFT は DSFT の周波数標本（一次元 DFT の行列表現）
一次元 $x[n]=(2,2)$ の DSFT $X(\mathrm{e}^{\mathrm{j}\omega})=2+2\mathrm{e}^{-\mathrm{j}\omega}=4\cos(\omega/2)\mathrm{e}^{-\mathrm{j}\omega/2}$ と，$N$ 点 DFT（$\omega=2\pi k/N$ の標本）を比べる。

In [ ]:
w = np.linspace(0, 2 * np.pi, 721)
Xw = 2 + 2 * np.exp(-1j * w)
X2 = np.fft.fft([2, 2])                            # N = 2
X8 = np.fft.fft([2, 2], 8)                         # N = 8（零値を追加）
k8 = np.arange(8)
print("X2 =", X2)
assert np.array_equal(np.real(X2), [4, 0])         # スライドの値

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 4.6), layout="constrained")
ax.plot(w, np.real(Xw), color=ccool, lw=2.2, label="$\\mathrm{Re}\\,X(\\mathrm{e}^{\\mathrm{j}\\omega})$")
ax.plot(w, np.imag(Xw), color=cwarm, lw=2.2, label="$\\mathrm{Im}\\,X(\\mathrm{e}^{\\mathrm{j}\\omega})$")
ax.plot(2 * np.pi * k8 / 8, np.real(X8), "o", mfc="none", color=ccool, ms=6, mew=1.8,
        label="$\\mathrm{Re}\\,X[k]\\ (N=8)$")
ax.plot(2 * np.pi * k8 / 8, np.imag(X8), "o", mfc="none", color=cwarm, ms=6, mew=1.8,
        label="$\\mathrm{Im}\\,X[k]\\ (N=8)$")
ax.plot(2 * np.pi * np.arange(2) / 2, np.real(X2), "s", mfc="none", color=cmain, ms=11, mew=2.5,
        label="$X[k]\\ (N=2)$")
ax.set(xlim=(0, 2 * np.pi), ylim=(-2.6, 4.8), xticks=np.arange(5) * np.pi / 2,
       xticklabels=["$0$", "$\\pi/2$", "$\\pi$", "$3\\pi/2$", "$2\\pi$"], yticks=[-2, 0, 2, 4], xlabel="$\\omega$")
ax.grid(True)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=9)   # 凡例は下の外
plt.show()

教科書の例「一次元 DFT の行列表現」。回転子 $W_N=\mathrm{e}^{-\mathrm{j}2\pi/N}$ を並べた $\mathbf{W}_N=(W_N^{kn})$ により $\tilde{\mathbf{x}}=\mathbf{W}_N\mathbf{x}$。IDFT は $\mathbf{W}_N^{-1}=\mathbf{W}_N^{\mathsf{H}}/N$。$N=2$ で確かめる。

In [ ]:
Nd = 2
kk, nn = np.meshgrid(np.arange(Nd), np.arange(Nd), indexing="ij")
WN = np.exp(-1j * 2 * np.pi / Nd) ** (kk * nn)     # DFT 行列 W_2
xv = np.array([2, 2])
Xv = WN @ xv                                       # W_2 x
print(np.round(np.real(WN)))
print("W_2 x =", Xv)
assert np.max(np.abs(Xv - np.fft.fft(xv))) < 1e-12
assert np.max(np.abs(np.linalg.inv(WN) - WN.conj().T / Nd)) < 1e-12   # W^{-1} = W^H/N
assert np.array_equal(np.round(np.real(WN)), [[1, 1], [1, -1]]) and np.allclose(Xv, [4, 0])   # スライドの値

## FFT の演算量
$N$ 点 DFT を行列演算で計算すると複素乗算は $N^2$ 回，周波数分割型 FFT では $\frac{N}{2}\log_2 N$ 回。

In [ ]:
N = 2 ** np.arange(1, 11)
mulMat = N ** 2
mulFFT = N // 2 * np.log2(N).astype(int)
fig, ax = plt.subplots(figsize=(4.2, 3.4), layout="constrained")
ax.loglog(N, mulMat, "o-", color=cwarm, lw=2.2, ms=5, label="行列演算 $N^2$")
ax.loglog(N, mulFFT, "s-", color=cmain, lw=2.2, ms=5, label="FFT $(N/2)\\log_2 N$")
ax.set(xlim=(2, 1024), ylim=(1, 1e8), xticks=[2, 8, 32, 128, 512], xticklabels=["2", "8", "32", "128", "512"],
       yticks=10.0 ** np.arange(0, 9, 2), xlabel="$N$", ylabel="複素乗算回数")   # 左上を凡例用に空ける
ax.grid(True)
ax.legend(loc="upper left", fontsize=10)
plt.show()

In [ ]:
ratio = mulMat[-1] / mulFFT[-1]
print(f"N = 1024：行列演算 {mulMat[-1]:,} 回，FFT {mulFFT[-1]:,} 回，比 {ratio:.0f}")
assert (mulMat[-1], mulFFT[-1], f"{ratio:.0f}") == (1048576, 5120, "205")   # スライドの値

## 例題：周波数スペクトル（シフトした sinc 配列）
教科書の例題。sinc 配列 $h[\boldsymbol{n}]=\prod_{d}\frac{\omega_{\mathrm{c}d}}{\pi}\mathrm{sinc}\left(\frac{\omega_{\mathrm{c}d}}{\pi}n_d\right)$ を $\boldsymbol{n}_0$ だけシフトした配列の DSFT は $H_{\boldsymbol{n}_0}(\mathrm{e}^{\mathrm{j}\boldsymbol{\omega}^\top})=\mathrm{e}^{-\mathrm{j}\boldsymbol{\omega}^\top\boldsymbol{n}_0}\prod_d\mathrm{rect}(\cdot)$ で，振幅スペクトルは rect の積（シフトで不変），位相スペクトルは平面 $-\boldsymbol{\omega}^\top\boldsymbol{n}_0$。

数値例：$\omega_{\mathrm{c}1}=\omega_{\mathrm{c}2}=\pi/2$，$\boldsymbol{n}_0=(1\ 2)^\top$。無限に広がる sinc 配列を $\boldsymbol{n}-\boldsymbol{n}_0\in\{-M,\ldots,M\}^2$ で打ち切り，DSFT を定義の和で計算する（打ち切りのため振幅には小さな波打ちが残る）。`np.sinc` は正規化 sinc 関数 $\sin(\pi x)/(\pi x)$ である。

In [ ]:
wc = np.array([np.pi / 2, np.pi / 2])              # 遮断角周波数 ω_c1, ω_c2
n0 = np.array([1, 2])                              # シフト量 n0
M = 200
m = np.arange(-M, M + 1)                           # n - n0 の範囲
wg = np.linspace(-np.pi, np.pi, 121)               # 周波数の標本（1 軸あたり）
h1 = wc[0] / np.pi * np.sinc(wc[0] / np.pi * m)    # 第 1 次元の sinc
h2 = wc[1] / np.pi * np.sinc(wc[1] / np.pi * m)    # 第 2 次元の sinc
hsh = np.outer(h1, h2)                             # h[n - n0]（行が n1，列が n2）
E1 = np.exp(-1j * np.outer(wg, m + n0[0]))         # e^{-j ω1 n1}，n1 = m + n0(1)
E2 = np.exp(-1j * np.outer(wg, m + n0[1]))         # e^{-j ω2 n2}
Hsh = E1 @ hsh @ E2.T                              # H_{n0}(e^{jω^T})（行が ω1，列が ω2）
Wg2, Wg1 = np.meshgrid(wg, wg)
passb = (np.abs(Wg1) < wc[0]) & (np.abs(Wg2) < wc[1])   # 通過域
magPass = np.array([np.min(np.abs(Hsh[passb])), np.max(np.abs(Hsh[passb]))])   # 通過域の振幅（≈1）
phErr = np.angle(Hsh * np.exp(1j * (Wg1 * n0[0] + Wg2 * n0[1])))
maxPhErr = np.max(np.abs(phErr[passb]))            # 通過域の位相は -ω^T n0 に一致
print("通過域の振幅:", magPass)
print("maxPhErr =", maxPhErr)

通過域の位相スペクトルを展開（アンラップ）して曲面として描くと，原点をとおる平面 $-\boldsymbol{\omega}^\top\boldsymbol{n}_0$ になる。sinc 配列は偶関数なので $h[\boldsymbol{n}-\boldsymbol{n}_0]$ は $\mathbf{c}=\boldsymbol{n}_0$ に関して対称で，群遅延 $\tau_\mathrm{g}(\boldsymbol{\omega})=-\nabla_{\boldsymbol{\omega}}\angle H_{\boldsymbol{n}_0}(\mathrm{e}^{\mathrm{j}\boldsymbol{\omega}^\top})$ は $\boldsymbol{n}_0$ で一定（直線位相）。スライドでは「フィルタの位相応答による分類」の二次元の直線位相の図に使う。

In [ ]:
ip = np.abs(wg) < wc[0]                            # 通過域の周波数（両軸とも同じ）
wp = wg[ip]
P1 = np.unwrap(np.angle(Hsh[np.ix_(ip, ip)]), axis=1)   # 各行（ω2 方向）を展開
col = np.unwrap(P1[:, 0])                          # 左端の列（ω1 方向）を展開
Pun = P1 + (col - P1[:, 0])[:, None]               # 行ごとのずれ（2π の整数倍）をそろえる
c0 = np.argmin(np.abs(wp))                         # ω = 0 の位置
Pun = Pun - 2 * np.pi * np.round(Pun[c0, c0] / (2 * np.pi))   # ω = 0 で位相 0
Wp2, Wp1 = np.meshgrid(wp, wp)
planeErr = np.max(np.abs(Pun + (Wp1 * n0[0] + Wp2 * n0[1])))   # 平面 -ω^T n0 との差
g1, g2 = np.gradient(Pun, wp[1] - wp[0])           # 位相の勾配（g1：ω1 方向，g2：ω2 方向）
tauG = -np.array([g1.mean(), g2.mean()])           # 群遅延 ≈ n0^T
print("planeErr =", planeErr)
print("τg =", tauG)
print("n0 =", n0)

In [ ]:
fig = plt.figure(figsize=(5.5, 5), layout="constrained")
ax = fig.add_subplot(projection="3d", computed_zorder=False)   # 描いた順に重ねる（原点の点を最前面に）
sk = np.r_[np.arange(0, wp.size, 7), wp.size - 1]  # 網目は 7 点おきに描く
wl2 = wc[0] * np.array([-1, 1, 1, -1]); wl1 = wc[1] * np.array([-1, -1, 1, 1])
ax.add_collection3d(Poly3DCollection([list(zip(wl2, wl1, np.zeros(4)))], facecolor=(*cgray, 0.15),
                                     edgecolor=cgray))   # ω1-ω2 平面（位相 0）
ax.plot_surface(Wp2, Wp1, Pun, color=tuple(1 - 0.35 * (1 - cm)), lw=0, alpha=0.85, shade=False)
ax.plot_wireframe(Wp2[np.ix_(sk, sk)], Wp1[np.ix_(sk, sk)], Pun[np.ix_(sk, sk)], color=cmain, lw=0.8)
ax.scatter([0], [0], [0], color=cwarm, s=60, depthshade=False)
ax.view_init(elev=20, azim=-70)                    # 勾配の向き (ω2,ω1)=(2,1) が画面の横になる向き
ax.set(xlim=wc[1] * np.array([-1, 1]), ylim=wc[0] * np.array([-1, 1]), zlim=np.array([-1.6, 1.6]) * np.pi,
       xticks=[-np.pi / 2, 0, np.pi / 2], xticklabels=["$-\\pi/2$", "$0$", "$\\pi/2$"],
       yticks=[-np.pi / 2, 0, np.pi / 2], yticklabels=["$-\\pi/2$", "$0$", "$\\pi/2$"],
       zticks=[-np.pi, 0, np.pi], zticklabels=["$-\\pi$", "$0$", "$\\pi$"],
       xlabel="$\\omega_2$", ylabel="$\\omega_1$",
       zlabel="$\\angle H_{\\mathbf{n}_0}(\\mathrm{e}^{\\mathrm{j}\\boldsymbol{\\omega}^\\top})$")
ax.set_box_aspect((1, 1, 1.1), zoom=0.9)
ax.tick_params(axis="y", pad=16)                   # 隅で ω2 の目盛と重ならないよう離す
ax.yaxis.labelpad = 22
plt.show()

## 例題：周波数応答（$3\times3$ 矩形フィルタ）
教科書の例題。$3\times3$ 矩形フィルタ（インパルス応答 $h[\boldsymbol{n}]=1/9,\ \boldsymbol{n}\in\{-1,0,1\}^2$）の周波数応答は

$$H(\mathrm{e}^{\mathrm{j}\boldsymbol{\omega}^\top})=\frac19(1+2\cos\omega_1)(1+2\cos\omega_2)$$

数値で確かめる：直流 $\boldsymbol{\omega}=\mathbf{0}$，最高周波数 $\boldsymbol{\omega}=(\pi\ \pi)^\top$，零点（$1+2\cos\omega_d=0$）。

In [ ]:
def Hsep(w1, w2):
    return (1 + 2 * np.cos(w1)) * (1 + 2 * np.cos(w2)) / 9


H00 = Hsep(0, 0)
Hpp = Hsep(np.pi, np.pi)
wz = np.arccos(-1 / 2)                             # 零点 ω_d = ±2π/3
Hz = Hsep(wz, 0)
print(f"H(0,0) = {H00:g}，H(π,π) = {Hpp:.3f} = {fracstr(Hpp)}，零点 ω = {pistr(wz)}，H(ω_z,0) = {Hz:g}")
assert (f"{H00:g}", f"{Hpp:.3f}", fracstr(Hpp), pistr(wz)) == ("1", "0.111", "1/9", "2π/3")   # スライドの値

4 種類のカーネルの振幅応答 $|H(\mathrm{e}^{\mathrm{j}\boldsymbol{\omega}^\top})|$ を DSFT の定義に従って計算する。カーネルの行の添え字が $n_1$（垂直），列の添え字が $n_2$（水平）で，中心が原点。

In [ ]:
W2, W1 = np.meshgrid(np.linspace(-np.pi, np.pi, 25), np.linspace(-np.pi, np.pi, 25))   # 横軸 ω2，縦軸 ω1（網目は粗めにして線を見やすく）
kers = [np.ones((3, 3)) / 9, np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16,
        np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]]), np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])]
ttl = ["平均（矩形）", "加重平均", "4 近傍ラプラシアン", "アンシャープマスク"]
m2, m1 = np.meshgrid(np.arange(-1, 2), np.arange(-1, 2))   # m1：行（垂直），m2：列（水平）
Hmag = []
for ker in kers:
    H = np.zeros(W1.shape, dtype=complex)
    for c, a1, a2 in zip(ker.ravel(), m1.ravel(), m2.ravel()):
        H += c * np.exp(-1j * (W1 * a1 + W2 * a2))
    Hmag.append(np.abs(H))

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(9, 8), layout="constrained", subplot_kw={"projection": "3d"})
for ax, Hm, tk in zip(axs.ravel(), Hmag, ttl):
    mesh(ax, W2, W1, Hm, cmapMain, lw=1.0)
    ax.view_init(elev=30, azim=-125)
    ax.set(xticks=pit, xticklabels=pil, yticks=pit, yticklabels=pil, xlabel="$\\omega_2$", ylabel="$\\omega_1$",
           zlabel="$|H(\\mathrm{e}^{\\mathrm{j}\\boldsymbol{\\omega}^\\top})|$", title=tk)
    ax.set_box_aspect(None, zoom=0.9)
plt.show()

In [ ]:
errBox = np.max(np.abs(Hmag[0] - np.abs(Hsep(W1, W2))))   # 矩形フィルタは閉じた形と一致
usRange = [Hmag[3].min(), Hmag[3].max()]           # アンシャープマスクの |H| の範囲
print("errBox =", errBox)
print("アンシャープマスクの |H| の範囲:", [f"{v:g}" for v in np.round(usRange, 6)])
assert np.allclose(usRange, [1, 9])                # スライドの値

## 直線位相の必要性
2 つの周波数成分からなるパルスを，直線位相のシステム（純粋な遅延 $h[n]=\delta[n-5]$）と非直線位相のシステム（全域通過フィルタ）に通す。どちらも振幅応答は 1 だが，非直線位相では成分ごとに遅れが異なり波形がひずむ。

In [ ]:
n = np.arange(80)
env = np.exp(-((n - 25) / 7) ** 2)
xin = env * (np.sin(0.25 * np.pi * n) + np.sin(0.6 * np.pi * n))
ylin = np.r_[np.zeros(5), xin[:-5]]                # 直線位相（遅延）
a = 0.8
ynl = signal.lfilter([-a, 1], [1, -a], xin)        # 全域通過（|H|=1，非直線位相）
fig, axs = plt.subplots(3, 1, figsize=(5, 4.2), layout="constrained", sharex=True)
for ax, y, c, ts in zip(axs, (xin, ylin, ynl), ("k", cmain, cwarm),
                        ("入力", "直線位相の出力（同じ形）", "非直線位相の出力（ひずむ）")):
    ax.plot(n, y, color=c, lw=1.8)
    ax.set(xlim=(5, 55), ylim=(-2.2, 2.2), title=ts)   # パルスのある範囲
    ax.grid(True)
axs[-1].set_xlabel("$n$")
plt.show()

位相応答 $\angle H(\mathrm{e}^{\mathrm{j}\omega})$ を比べる。直線位相は $-5\omega$ の直線，全域通過フィルタは曲線。

In [ ]:
wl = np.linspace(0, np.pi, 256)
_, Hn = signal.freqz([-a, 1], [1, -a], worN=wl)
fig, ax = plt.subplots(figsize=(4.2, 3.6), layout="constrained")
ax.plot(wl, -5 * wl, color=cmain, lw=2.4, label="直線位相（傾き -5）")
ax.plot(wl, np.unwrap(np.angle(Hn)), color=cwarm, lw=2.4, label="非直線位相")
ax.set(xlim=(0, np.pi), xticks=[0, np.pi / 2, np.pi], xticklabels=["$0$", "$\\pi/2$", "$\\pi$"],
       xlabel="$\\omega$", ylabel="$\\angle H(\\mathrm{e}^{\\mathrm{j}\\omega})$")
ax.grid(True)
ax.legend(loc="center right", fontsize=9)
plt.show()

## 画像信号の振幅スペクトルと位相スペクトル
教科書のサンプル画像 msipimg04（石造りの建物）をグレースケールにして $256\times256$ 画素に縮小し，DFT を求めて振幅スペクトル（対数表示，原点を中央に移動）と位相スペクトルを表示する。アーチや柱の輪郭がはっきりした画像なので，位相だけからの再構成で輪郭が残る様子がよく分かる（msipimg05 の石像，06 の路面，01 の海岸と見比べて選んだ）。

In [ ]:
Xc = img_as_float(vie.msipimg(4, 256, "gray"))   # msipimg04（石造りの建物）
Fc = np.fft.fft2(Xc)
magS = np.log(1 + np.abs(np.fft.fftshift(Fc)))
phaS = np.angle(np.fft.fftshift(Fc))
fig, axs = plt.subplots(1, 3, figsize=(10.5, 3.8), layout="constrained")
show(axs[0], Xc, "画像")
show(axs[1], magS / magS.max(), "振幅スペクトル（対数表示）")
show(axs[2], (phaS + np.pi) / (2 * np.pi), "位相スペクトル")
plt.show()

振幅だけ（位相 0）と位相だけ（振幅 1）から IDFT で再構成する。位相だけからの再構成の方が輪郭の面影を残す。

In [ ]:
Ymag = np.real(np.fft.ifft2(np.abs(Fc)))           # 位相 0
Ypha = np.real(np.fft.ifft2(np.exp(1j * np.angle(Fc))))   # 振幅 1
Ymag = np.fft.fftshift(Ymag)                       # 原点付近に集まるので中央へ移動して表示
Lmag = np.log(1 + np.abs(Ymag))


def minmax(A):
    """最小値を 0，最大値を 1 に写す（表示用）"""
    return (A - A.min()) / (A.max() - A.min())


fig, axs = plt.subplots(1, 2, figsize=(8, 4.2), layout="constrained")
show(axs[0], minmax(Lmag), "振幅スペクトルのみ")   # 最小〜最大を黒〜白に
show(axs[1], minmax(Ypha), "位相スペクトルのみ")
plt.show()

## 直線位相と対称性
偶対称なインパルス応答 $h=(1,3,3,1)/8$（中心 $c=1.5$）は直線位相 $\angle H(\mathrm{e}^{\mathrm{j}\omega})=-1.5\omega$（群遅延 $\tau_\mathrm{g}=c$）。非対称な $h=(4,2,1,1)/8$ は非直線位相。

In [ ]:
hs = np.array([1, 3, 3, 1]) / 8; hn = np.array([4, 2, 1, 1]) / 8
cs = (hs.size - 1) / 2                             # 対称の中心
print("c =", cs)
assert np.array_equal(hs, hs[::-1])
_, Hs = signal.freqz(hs, 1, worN=wl)
_, Hnn = signal.freqz(hn, 1, worN=wl)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3.6), layout="constrained", gridspec_kw={"width_ratios": [1, 2]})
ax = axs[0]
ml, sl, _ = ax.stem(np.arange(4), hs, basefmt=" ", label="偶対称")
plt.setp(ml, color=cmain, ms=5); plt.setp(sl, color=cmain, lw=2)
ml, sl, _ = ax.stem(np.arange(4), hn, basefmt=" ", label="非対称")
plt.setp(ml, color=cwarm, mfc="none", ms=5); plt.setp(sl, color=cwarm, lw=2)
ax.set(xlim=(-0.5, 3.5), ylim=(0, 0.85), xticks=range(4), yticks=np.arange(0, 0.7, 0.2),   # 上を凡例用に空ける
       title="インパルス応答", xlabel="$n$", ylabel="$h[n]$")
ax.grid(True)
ax.legend(loc="upper right", fontsize=8)
ax = axs[1]
ax.plot(wl[:-1], np.unwrap(np.angle(Hs[:-1])), color=cmain, lw=2.4, label="偶対称：直線位相")   # ω = π は H = 0 で位相が定まらないので除く
ax.plot(wl, np.unwrap(np.angle(Hnn)), color=cwarm, lw=2.4, label="非対称：非直線位相")
ax.set(xlim=(0, np.pi), ylim=(-6.8, 0.4), xticks=[0, np.pi / 2, np.pi], xticklabels=["$0$", "$\\pi/2$", "$\\pi$"],
       title="位相応答", xlabel="$\\omega$", ylabel="$\\angle H(\\mathrm{e}^{\\mathrm{j}\\omega})$")   # 左下を凡例用に空ける
ax.grid(True)
ax.legend(loc="lower left", fontsize=8)
plt.show()

In [ ]:
slope = np.mean(np.diff(np.unwrap(np.angle(Hs[:200]))) / np.diff(wl[:200]))   # 傾き ≈ -1.5
print(f"傾き = {slope:.1f}，対称の中心 c = {cs:g}")
assert f"{slope:.1f}" == "-1.5" and f"{cs:g}" == "1.5"   # スライドの値

## まとめ
- 二次元余弦波の角周波数は位相の各方向の偏微分で与えられ，標本化すると $\boldsymbol{\omega}=\boldsymbol{L}^\top\boldsymbol{\nu}$ となる
- 周期構造行列 $\boldsymbol{N}$ の基本周期に含まれる整数ベクトルの数は $|\det(\boldsymbol{N})|$
- DFT は DSFT の周波数標本で，行列 $\mathbf{W}_N$ で表され，各次元の一次元 DFT（FFT）に分離できる
- 位置ずれは位相スペクトルだけを変え，畳み込みは周波数領域では積になる（周波数応答）
- 画像では位相スペクトルが重要で，対称なインパルス応答は直線位相をもつ

© Copyright, Shogo MURAMATSU, All rights reserved.